<a href="https://colab.research.google.com/github/denisejroth/pixels-frames-transformers/blob/main/part2_vectors/07_bias_measurement_and_faces/notebooks/07_bias_measurement_and_faces_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pixels, Frames & Transformers
## Module 07: Bias, Measurement and Faces · Solutions

**A Self-Paced Course in Computational Image and Video Analysis**  
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

This is the **solutions** version. Every ✏️ exercise is filled in with one possible answer and a short comment. There is usually more than one good way to solve an exercise, and your numbers may differ a little from run to run, so read the comments as "what to look for" rather than exact results.

Vision models learn from the images they were trained on, and those images over-represent some
people, places and situations. The result is errors that are not random: a classifier that works
well on average can fail far more often for some groups. In this notebook you audit a model for
such errors, see what image embeddings encode about the people in them, and work out how errors
that differ between groups can bias the conclusions of a study.

In this notebook you will:

- train a classifier on a deliberately skewed sample of faces and audit it on a balanced set, in
  the spirit of the *Gender Shades* audit
- report error rates per group with confidence intervals, including for intersecting groups
- test what embeddings of the Congress portraits reveal about the members and about the photos
- think through skin tone scales versus racial categories, and what not to infer from a face
- simulate how measurement error that differs by group creates a group difference that isn't
  there, and correct for it with a validation sample

**Data.** [FairFace](https://huggingface.co/datasets/HuggingFaceM4/FairFace) (Kärkkäinen & Joo,
2021, CC BY 4.0), about 100,000 face images collected mostly from public Flickr photos, built to
be balanced across seven race groups, gender and age. Its labels were assigned by crowd workers who looked at each
face. Then the Congress portraits from module 01.

**A note on purpose.** We train a gender classifier below for one reason: to audit it, as
Buolamwini and Gebru (2018) audited commercial systems. FairFace's gender labels are binary and
record what annotators *perceived*; its race labels are annotators' guesses at social categories.
Neither says anything about how a person identifies. Don't use a model like this to label real
people.

**Runtime.** A GPU is recommended: *Runtime → Change runtime type → T4 GPU*. On a CPU the notebook
uses a smaller sample of faces.

## 0. Setup

We load FairFace with the Hugging Face `datasets` library and compute DINOv2 embeddings with the
helper from module 06 (`course_torch.embed_images`), which saves them to a file so we only compute
them once.

In [ ]:
!pip install -q datasets

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import torch
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split

SEED = 42
rng = np.random.default_rng(SEED)

HEADERS = {"User-Agent": "PixelsFramesTransformers/1.0 (https://github.com/denisejroth/pixels-frames-transformers) course-notebook"}
BASE = "https://raw.githubusercontent.com/denisejroth/pixels-frames-transformers/main/shared/"
for helper in ["course_data.py", "course_torch.py"]:
    if not Path(helper).exists():
        response = requests.get(BASE + helper, headers=HEADERS, timeout=30)
        response.raise_for_status()  # stop if the download failed, rather than saving an error page
        Path(helper).write_text(response.text)
import course_data
import course_torch
from course_torch import DEVICE, load_rgb

ON_GPU = DEVICE == "cuda"
print("Running on:", DEVICE)

## 1. Who is in the training data?

Most large image collections were scraped from the web, and the web over-represents some people:
lighter-skinned people, people in North America and Europe, people photographed in particular
settings. Buolamwini and Gebru (2018) showed that two benchmarks widely used for face analysis
were mostly made up of lighter-skinned faces, and that commercial gender classifiers made far
more errors on darker-skinned women than on lighter-skinned men.

Bias also shows up in *what* a model says about people. Schwemmer and colleagues (2020) sent
photos of members of the US Congress to commercial image recognition services and found that
images of women received far more labels about appearance (such as hairstyle or smile), and images
of men more labels about occupation and status (such as official or businessperson).

FairFace was built in response to skewed data: its creators collected faces until the seven race
groups they used were roughly balanced. That makes it useful for exactly one thing we need here: checking how
a model performs **for each group**. We use the validation split (about 11,000 faces) and save
the images to disk, so the embedding helper can read them.

In [ ]:
fairface = load_dataset("HuggingFaceM4/FairFace", "0.25", split="validation")
RACES = fairface.features["race"].names
GENDERS = fairface.features["gender"].names
AGES = fairface.features["age"].names

N_FACES = len(fairface) if ON_GPU else min(3000, len(fairface))     # fewer faces on a CPU
keep = np.sort(rng.choice(len(fairface), N_FACES, replace=False))

FACE_DIR = Path("fairface")
FACE_DIR.mkdir(exist_ok=True)
rows = []
for i in keep:
    item = fairface[int(i)]
    path = FACE_DIR / f"{i}.jpg"
    if not path.exists():
        item["image"].convert("RGB").save(path, quality=95)
    rows.append({"path": str(path), "race": RACES[item["race"]], "gender": GENDERS[item["gender"]],
                 "age": AGES[item["age"]]})
faces = pd.DataFrame(rows)
print(f"{len(faces)} faces")
pd.crosstab(faces["race"], faces["gender"], margins=True)

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(13, 1.6))
for ax, path in zip(axes, faces["path"].sample(10, random_state=SEED)):
    ax.imshow(load_rgb(path))
    ax.axis("off")
plt.show()

In [ ]:
face_emb = course_torch.embed_images(faces["path"], cache="fairface_dinov2.npz")
print("Embeddings:", face_emb.shape)

## 2. Train on skewed data, audit on balanced data

We split the faces in two halves. The **audit half** stays balanced and is only used for
evaluation. From the other half, the **pool**, we draw a training sample that is as skewed as many
real training sets: mostly faces labeled White, very few labeled Black. Within each group, half
the faces are labeled Male and half Female, so the skew is in who appears, not in the outcome.

Then we train a logistic regression on the DINOv2 embeddings to predict the (perceived) gender
label, exactly as we trained the linear probe in module 06.

This is one of two common audit designs. The other is a **paired design**: show a model images
that are identical except for one attribute and see whether its output changes. Module 18 uses
that design to audit image generators.

In [ ]:
faces["stratum"] = faces["race"] + " / " + faces["gender"]
pool_idx, audit_idx = train_test_split(np.arange(len(faces)), test_size=0.5, stratify=faces["stratum"], random_state=SEED)
pool, audit = faces.iloc[pool_idx], faces.iloc[audit_idx]

SKEWED = {"White": 0.60, "East Asian": 0.15, "Latino_Hispanic": 0.08, "Indian": 0.06,
          "Middle Eastern": 0.05, "Southeast Asian": 0.04, "Black": 0.02}
TRAIN_SIZE = 1200 if ON_GPU else 500

def draw_training_sample(pool, shares, n=TRAIN_SIZE, seed=SEED):
    """Draw n faces from the pool with the given share per race group, half Male and half Female."""
    parts = []
    for race, share in shares.items():
        per_gender = round(n * share / 2)
        for gender in GENDERS:
            group = pool[(pool["race"] == race) & (pool["gender"] == gender)]
            parts.append(group.sample(min(per_gender, len(group)), random_state=seed))
    return pd.concat(parts)

skewed_train = draw_training_sample(pool, SKEWED)
print(skewed_train["race"].value_counts(normalize=True).round(3).rename("share of training faces"))

def fit_gender_classifier(train):
    return LogisticRegression(max_iter=5000).fit(face_emb[train.index], train["gender"])

skewed_model = fit_gender_classifier(skewed_train)
audit = audit.assign(pred_skewed=skewed_model.predict(face_emb[audit.index]))
print(f"\nOverall accuracy on the balanced audit set: {(audit['pred_skewed'] == audit['gender']).mean():.3f}")

A single overall number hides how errors are distributed. An audit reports the **error rate per
group**, with a confidence interval, because some groups are small and their error rates are
uncertain. We use the bootstrap: resample each group's faces with replacement many times and see
how much the error rate varies.

In [ ]:
def error_by_group(table, pred_col, group_cols, n_boot=2000):
    """Error rate per group with a 95% bootstrap interval."""
    out = []
    for key, g in table.groupby(group_cols):
        wrong = (g[pred_col] != g["gender"]).to_numpy()
        boots = rng.choice(wrong, size=(n_boot, len(wrong))).mean(axis=1)
        out.append({"group": key if isinstance(key, str) else " / ".join(key), "n": len(wrong),
                    "error": wrong.mean(), "low": np.percentile(boots, 2.5), "high": np.percentile(boots, 97.5)})
    return pd.DataFrame(out).sort_values("error").reset_index(drop=True)

def plot_errors(table, title, ax=None):
    ax = ax or plt.subplots(figsize=(6, 0.35 * len(table) + 1))[1]
    ax.errorbar(table["error"], table["group"], xerr=[table["error"] - table["low"], table["high"] - table["error"]],
                fmt="o", capsize=3)
    ax.set_xlabel("error rate (with 95% interval)")
    ax.set_title(title)
    ax.set_xlim(left=0)
    return ax

skewed_errors = error_by_group(audit, "pred_skewed", "race")
plot_errors(skewed_errors, "Trained on skewed data")
plt.show()
skewed_errors.round(3)

Look at the ratio between the highest and the lowest error rate, and at the intervals: do they
overlap? With embeddings as good as DINOv2's, overall accuracy is high, and the gaps can be
smaller than in the commercial systems of 2018. Whether they disappear is an empirical question,
and that is the point of an audit: you can't know without measuring.

> **✏️ Exercise 1**
>
> Does balancing the training data help?
>
> 1. Draw a **balanced** training sample of the same size (equal shares for all seven groups) with
>    `draw_training_sample`, train a second classifier, and add its predictions to `audit` as
>    `pred_balanced`.
> 2. Compute the error rates per group for both models and plot them side by side.
> 3. Did the gap between the best and the worst group shrink? Did overall accuracy change?

In [ ]:
# ✅ Solution
BALANCED = {race: 1 / len(SKEWED) for race in SKEWED}
balanced_train = draw_training_sample(pool, BALANCED)
balanced_model = fit_gender_classifier(balanced_train)
audit = audit.assign(pred_balanced=balanced_model.predict(face_emb[audit.index]))

balanced_errors = error_by_group(audit, "pred_balanced", "race")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharex=True)
plot_errors(skewed_errors, "Trained on skewed data", axes[0])
plot_errors(balanced_errors, "Trained on balanced data", axes[1])
plt.tight_layout()
plt.show()

for name, errors, col in [("skewed", skewed_errors, "pred_skewed"), ("balanced", balanced_errors, "pred_balanced")]:
    print(f"{name:<9} overall accuracy {(audit[col] == audit['gender']).mean():.3f}   "
          f"worst group {errors['error'].max():.3f}   best group {errors['error'].min():.3f}")

# Balancing usually lowers the error for the groups that were rare in the skewed sample and costs
# little or nothing overall, so the gap shrinks. It often doesn't vanish: some differences come
# from the pre-trained embeddings themselves (DINOv2's own training data), from image quality, or
# from the annotations. Balancing your training data can't fix what the backbone never learned.

> **✏️ Exercise 2**
>
> Gender Shades found the largest errors not for a race group or a gender on its own, but where the
> two **intersect**. Compute the error rates of the skewed model for every race × gender
> combination (`group_cols=["race", "gender"]`) and plot them. Which subgroup has the highest error
> rate? How wide are the intervals, and what does that mean for how many faces an audit needs?

In [ ]:
# ✅ Solution
intersection = error_by_group(audit, "pred_skewed", ["race", "gender"])
plot_errors(intersection, "Skewed model: race x gender")
plt.show()
print(intersection.tail(4).round(3))

# The worst subgroups are often combinations that were rare in training (Black women made up about
# 1% of the skewed sample), and their errors can be larger than the race or gender averages suggest. The
# intervals are wider than for the seven groups, because each subgroup has about half as many
# faces. To detect a difference of a few percentage points between subgroups, you need hundreds
# of faces per subgroup; with a few dozen, only very large gaps are distinguishable from noise.

## 3. What do embeddings encode about the people in them?

In module 06 you used embeddings as general-purpose features. Any classifier you train on them can
draw on everything they encode, including things you never meant it to use. A simple way to see
what is in there is a **probe**: try to predict an attribute from the embeddings with
cross-validation. If you can, the information is present, and a downstream model could pick it up
as a shortcut.

We embed the official portraits of all current members of Congress and probe for four
attributes taken from the congress-legislators metadata, not inferred from faces. Note that we
are not building tools to infer these things: we are checking what the vectors carry.

In [ ]:
portraits = course_data.get_congress_portraits(n=None)
portraits = portraits[portraits["party"].isin(["Democrat", "Republican"])].reset_index(drop=True)
portrait_emb = course_torch.embed_images(portraits["local_path"], cache="portraits_dinov2.npz")

targets = {
    "party (Democrat or Republican)": portraits["party"],
    "gender (from the metadata)": portraits["gender"],
    "chamber (House or Senate)": portraits["chamber"],
    "entered Congress in 2017 or later": portraits["first_year"] >= 2017,
}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
for name, y in targets.items():
    pred = cross_val_predict(LogisticRegression(class_weight="balanced", max_iter=5000), portrait_emb, y, cv=cv)
    print(f"{name:<36} balanced accuracy {balanced_accuracy_score(y, pred):.2f}   (chance: 0.50)")

> **✏️ Exercise 3**
>
> Look at the four results.
>
> 1. Which attributes can be predicted well above chance from the portraits? For each, what in the
>    **photo** could carry that information?
> 2. Suppose you trained a classifier on these embeddings to code "formal versus casual
>    self-presentation" in politicians' Instagram photos. Which of these attributes might it learn
>    to rely on, and how would you check?

**✅ One possible answer**

1. Gender (from the metadata) is usually predicted very well: the embeddings encode appearance,
   and the classifier picks up visual cues correlated with the recorded gender. When members
   entered Congress is often predictable too, but through the **photos**: portraits from the same
   years share backdrops, lighting and style (remember the dark portraits of 2017 to 2021 in
   module 01). Chamber can be predictable for the same reason: House and Senate portraits are
   produced differently. Party is often predicted above chance without any "party look": it
   correlates with gender, age and the years members entered, and with flags and backdrops chosen
   by offices.
2. A "formal versus casual" classifier could quietly learn gender (if your coded examples of
   formal photos are mostly of men, say), photo era or production style. Check by auditing it:
   compare its outputs across gender, party and time **within** sets of images you coded the same
   way, and see whether predictions shift with attributes that should not matter. Balance your
   coded sample across those attributes, and report error rates by group, as in section 2.

## 4. Skin tone, racial categories, and what not to infer from a face

The audit in section 2 grouped faces by FairFace's race labels. Those labels are **social
categories assigned by annotators** who looked at a face: they are not self-identification, and
the categories themselves are one particular, contestable scheme. Two alternatives are common in
audits:

- **Skin tone scales** describe appearance rather than identity. Buolamwini and Gebru (2018) used
  the six-point Fitzpatrick scale from dermatology; the ten-point **Monk Skin Tone scale**,
  developed by sociologist Ellis Monk and released by Google in 2022, covers darker skin tones in
  more detail. Skin tone is closer to what a camera and a model actually "see", and it lets you
  test, for instance, whether errors grow with darker skin and poor lighting.
- **Self-reported attributes**, collected from people with their consent, measure identity
  directly, but are rarely available for images collected from the web.

Inferring gender, race, age or emotion *from a face* is a measurement problem as much as an
ethical one. Appearance is a noisy, biased proxy for identity; the error is larger for some groups
(as section 2 showed); and the categories a model outputs are the ones its labelers used.
Alternatives worth considering: use metadata or self-reports where they exist; code the
**context** (who is speaking, which role a person has, how they are framed) rather than the person's
identity; or, if you must, have trained human coders code perceived attributes with a codebook and
report their agreement.

> **✏️ Exercise 4**
>
> A colleague wants to study whether news photos show women politicians smiling more often than men.
> They plan to detect faces, classify gender and smiling with off-the-shelf models, and compare the
> rates.
>
> Name three ways this measurement could go wrong, and propose a better design for each.

**✅ One possible answer**

1. **Who is a woman politician is inferred from appearance.** Gender classification is binary,
   error-prone and more error-prone for some groups. *Better:* identify the politicians (captions,
   names in the article, a list of officeholders) and take gender from public records, which also
   lets you analyze individuals rather than faces.
2. **The smile detector's errors may differ by gender** (for example, if it was trained on more
   faces of one gender, or responds to makeup or facial hair). A difference in detected smiles
   could then come from the detector. *Better:* hand-code smiling on a sample stratified by
   gender, measure the detector's error rates per group, and correct the estimates (section 5).
3. **Photo selection, not people, may differ.** Editors choose which photo to run; women may be
   photographed in different settings or roles. *Better:* sample photos comparably (same outlets,
   events, roles), record context variables, and be clear that the study measures how photos
   *portray* politicians, not how politicians behave.

## 5. When errors differ by group, conclusions shift

Why does an audit matter for social science, beyond fairness? Because most studies use a
classifier's output as a **measurement**, and then compare groups. If the classifier's errors
differ between the groups you compare, the comparison is biased, even when the classifier is
accurate overall (TeBlunthuis et al., 2024).

A simulation makes this concrete. Take 4,000 photos of politicians from two groups, A and B. In
truth, 40% of the people in **both** groups are smiling. A smile detector finds 90% of smiles in
group A but only 70% in group B, and wrongly reports a smile in 10% of non-smiling faces in both.

In [ ]:
n = 4000
group = np.where(rng.random(n) < 0.5, "A", "B")
smiling = rng.random(n) < 0.40                                   # same true rate in both groups
sensitivity = np.where(group == "A", 0.90, 0.70)                 # share of smiles detected
false_positive = 0.10                                            # share of non-smiles called a smile
detected = np.where(smiling, rng.random(n) < sensitivity, rng.random(n) < false_positive)

sim = pd.DataFrame({"group": group, "smiling": smiling, "detected": detected})
summary = sim.groupby("group")[["smiling", "detected"]].mean().round(3)
summary.columns = ["true rate", "measured rate"]
print(summary)
diff_true = summary.loc["A", "true rate"] - summary.loc["B", "true rate"]
diff_measured = summary.loc["A", "measured rate"] - summary.loc["B", "measured rate"]
print(f"\nDifference A - B: true {diff_true:+.3f}, measured {diff_measured:+.3f}")
print(f"Overall accuracy of the detector: {(sim['smiling'] == sim['detected']).mean():.2f}")

The detector is right about 86% of the time, yet the measured rates suggest that group A smiles
noticeably more than group B, a difference that doesn't exist. A regression on the detector's
output would report it as a significant effect.

The fix starts with a **validation sample**: hand-code a random sample of photos from each group,
and estimate the detector's sensitivity and false positive rate per group. With those, you can
correct a measured rate (Rogan & Gladen, 1978):

`corrected rate = (measured rate - false positive rate) / (sensitivity - false positive rate)`

> **✏️ Exercise 5**
>
> 1. Draw a validation sample of 200 photos per group from `sim`, as if you had hand-coded them (in
>    the simulation, `smiling` plays the role of your codes).
> 2. In each group's sample, estimate the sensitivity (the share of true smiles that were detected)
>    and the false positive rate (the share of non-smiles that were detected).
> 3. Correct each group's measured rate with the formula above. How close do the corrected rates
>    and their difference get to the truth? Rerun with validation samples of 50 per group: what
>    happens?

In [ ]:
# ✅ Solution
def corrected_rates(sim, n_per_group, seed=SEED):
    out = {}
    for g, data in sim.groupby("group"):
        sample = data.sample(n_per_group, random_state=seed)
        sens = sample.loc[sample["smiling"], "detected"].mean()
        fpr = sample.loc[~sample["smiling"], "detected"].mean()
        measured = data["detected"].mean()
        out[g] = {"measured": measured, "sensitivity": sens, "false positive": fpr,
                  "corrected": (measured - fpr) / (sens - fpr), "true": data["smiling"].mean()}
    return pd.DataFrame(out).T.round(3)

for size in [200, 50]:
    table = corrected_rates(sim, size)
    print(f"Validation sample of {size} per group")
    print(table)
    print(f"corrected difference A - B: {table.loc['A', 'corrected'] - table.loc['B', 'corrected']:+.3f}\n")

# With 200 coded photos per group, the corrected rates land close to the true 40% in both groups
# and the spurious difference mostly disappears. With 50 per group, the estimates of sensitivity
# and false positive rate are noisy, so the corrected rates jump around (rerun with other seeds to
# see it). The correction is only as good as the validation sample, and its uncertainty should be
# carried into your final estimates, for example by bootstrapping the whole procedure.
# TeBlunthuis et al. (2024) discuss regression methods that do this properly.

## 6. What to report

When a measurement in your study involves people in images, readers need to know:

- **who is in your data**, and how that compares to the population you make claims about;
- **how labels about people were obtained**: metadata, self-reports, or someone's perception (and
  whose), with the categories used;
- **the method's error rates per relevant group**, with intervals and the size of each group in
  your validation sample;
- **what you did about differences**: rebalancing, correction, or a limitation statement;
- **what you chose not to infer**, and why.

## Wrap-up

You have seen that:

- training data over-represent some people, and a model trained on skewed data can make more
  errors for the groups it rarely saw;
- an audit reports error rates per group and per intersection, with intervals, on a balanced
  evaluation set;
- rebalancing training data narrows gaps but can't remove what the pre-trained backbone carries;
- embeddings encode information about people and about how photos were produced, which any
  downstream model can use as a shortcut;
- race labels from annotators, skin tone and self-identification measure different things, and
  inferring identity from faces is a measurement problem as much as an ethical one;
- errors that differ between groups bias group comparisons, and a validation sample per group lets
  you measure and correct that.

**Next:** Part 3 starts with module 08, which explains the vision transformers inside DINOv2 and
most of the models that follow.